In [1]:
import pandas as pd

# Define the file path
file_path = '/Volumes/T7/bid-merged-fcas-2017-2018'

# Read the Parquet file into a DataFrame
df = pd.read_parquet(file_path)

# Display general information about the DataFrame
print("DataFrame Info:")
df.info()

# Show the first few rows of the DataFrame
print("\nDataFrame Head:")
print(df.head())

# Show the last few rows of the DataFrame
print("\nDataFrame Tail:")
print(df.tail())

# Display descriptive statistics for numerical columns (and object columns if needed)
print("\nDataFrame Description:")
print(df.describe(include='all'))

# List all column names
print("\nDataFrame Columns:")
print(df.columns.tolist())

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 111497760 entries, 0 to 111497759
Data columns (total 32 columns):
 #   Column                        Dtype         
---  ------                        -----         
 0   SETTLEMENTDATE                datetime64[ns]
 1   DUID                          object        
 2   BIDTYPE                       object        
 3   OFFERDATE_volume              object        
 4   MAXAVAIL                      object        
 5   ENABLEMENTMIN                 object        
 6   ENABLEMENTMAX                 object        
 7   LOWBREAKPOINT                 object        
 8   HIGHBREAKPOINT                object        
 9   INTERVAL_DATETIME             object        
 10  Participant                   object        
 11  Station Name                  object        
 12  Region                        object        
 13  Dispatch Type                 object        
 14  Category                      object        
 15  Classificati

In [2]:
# Get all unique DUIDs
unique_duids = df["DUID"].unique()

# Display count and values
print(f"Total unique DUIDs: {len(unique_duids)}")
print("\nUnique DUIDs:")
print(unique_duids)

# Optional: Save to a file
# pd.Series(unique_duids).to_csv('unique_duids.csv', index=False, header=['DUID'])

Total unique DUIDs: 11

Unique DUIDs:
['OSB-AG' 'QPS5' 'PPCCGT' 'TORRB1' 'TORRB4' 'TORRB2' 'TORRB3' 'HDWF2'
 'LONSDALE' 'HDWF3' 'HDWF1']


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Define the file path
file_path = '/Volumes/T7/bid-price-data-sorted-feather-B1/hpr1_combined_data.feather'

# Load the feather file
try:
    df_hpr1 = pd.read_feather(file_path)
    print(f"Successfully loaded data with {len(df_hpr1)} rows")
    
    # Display basic information
    print("\nDataFrame Info:")
    print(df_hpr1.info())
    
    # Show the first few rows
    print("\nSample Data:")
    print(df_hpr1.head())
    
    # Check unique values in key columns
    print("\nUnique DUIDs:", df_hpr1['DUID'].unique())
    print("\nUnique BIDTYPEs:", df_hpr1['BIDTYPE'].unique())
    
    if 'Participant' in df_hpr1.columns:
        print("\nParticipant(s):", df_hpr1['Participant'].unique())
    
    if 'Station Name' in df_hpr1.columns:
        print("\nStation Name(s):", df_hpr1['Station Name'].unique())
    
    # Date range information
    print("\nDate Range:")
    print(f"Start: {df_hpr1['SETTLEMENTDATE'].min()}")
    print(f"End: {df_hpr1['SETTLEMENTDATE'].max()}")
    
    # If you want to save the plot:
    # plt.savefig('hpr1_bid_prices.png', dpi=300)
    
except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
    print("Please verify the file path and try again.")
except Exception as e:
    print(f"Error loading the file: {str(e)}")

In [17]:
import pandas as pd
import os
import glob
from datetime import datetime

# Define the folder path
folder_path = '/Volumes/T7/bid-price-data-sorted-feather-B1'

# Get all feather files in the folder
feather_files = glob.glob(os.path.join(folder_path, '*.feather'))
print(f"Found {len(feather_files)} feather files")

# List to store results
hpr1_results = []

# Process each file
for file_path in feather_files:
    print(f"Processing {os.path.basename(file_path)}...")
    
    try:
        # Read the feather file
        df = pd.read_feather(file_path)
        
        # Check if DUID and SETTLEMENTDATE columns exist
        if 'DUID' not in df.columns or 'SETTLEMENTDATE' not in df.columns:
            print(f"  Missing required columns in {os.path.basename(file_path)}")
            continue
        
        # Ensure SETTLEMENTDATE is datetime
        if not pd.api.types.is_datetime64_any_dtype(df['SETTLEMENTDATE']):
            df['SETTLEMENTDATE'] = pd.to_datetime(df['SETTLEMENTDATE'], errors='coerce')
        
        # Filter for dates after 2017 and DUID containing HPR1
        filtered_df = df[(df['SETTLEMENTDATE'] >= '2018-01-01') & 
                          (df['DUID'].str.contains('H', case=False, na=False))]
        
        # If we found matches, store the information
        if not filtered_df.empty:
            file_info = {
                'file_name': os.path.basename(file_path),
                'matches_count': len(filtered_df),
                'unique_duids': filtered_df['DUID'].unique().tolist(),
                'date_range': (filtered_df['SETTLEMENTDATE'].min(), 
                              filtered_df['SETTLEMENTDATE'].max())
            }
            hpr1_results.append(file_info)
            
            print(f"  Found {len(filtered_df)} matches with DUID: {filtered_df['DUID'].unique().tolist()}")
        else:
            print(f"  No matches found")
            
    except Exception as e:
        print(f"  Error processing {os.path.basename(file_path)}: {str(e)}")

# Display summary of results
print("\n=== SUMMARY ===")
print(f"Found HPR1 entries in {len(hpr1_results)} files")

for result in hpr1_results:
    print(f"\nFile: {result['file_name']}")
    print(f"Matches: {result['matches_count']}")
    print(f"DUIDs: {result['unique_duids']}")
    print(f"Date range: {result['date_range'][0]} to {result['date_range'][1]}")

# Optional: Combine all matching data
if hpr1_results:
    print("\nCombining all matching data...")
    combined_data = []
    
    for file_info in hpr1_results:
        df = pd.read_feather(os.path.join(folder_path, file_info['file_name']))
        
        # Ensure SETTLEMENTDATE is datetime
        if not pd.api.types.is_datetime64_any_dtype(df['SETTLEMENTDATE']):
            df['SETTLEMENTDATE'] = pd.to_datetime(df['SETTLEMENTDATE'], errors='coerce')
        
        # Filter for dates after 2017 and DUID containing HPR1
        filtered_df = df[(df['SETTLEMENTDATE'] >= '2018-01-01') & 
                          (df['DUID'].str.contains('H', case=False, na=False))]
        
        combined_data.append(filtered_df)
    
    # Combine all filtered data
    if combined_data:
        all_hpr1_data = pd.concat(combined_data, ignore_index=True)
        print(f"Combined data shape: {all_hpr1_data.shape}")
        
        # Save combined data to a new file
        output_path = os.path.join(folder_path, 'hpr1_combined_data.feather')
        all_hpr1_data.to_feather(output_path)
        print(f"Combined data saved to {output_path}")

Found 1453 feather files
Processing PUBLIC_DVD_BIDDAYOFFER_D_20210401.feather...
  Found 114 matches with DUID: ['BBTHREE2', 'COHUNSF1', 'HDWF1', 'HDWF2', 'HDWF3', 'HPRL1', 'MACKNTSH', 'SNOWNTH1', 'VSQHT1V1', 'VSSSH1S1', 'W/HOE#1', 'W/HOE#2', 'WHITSF1', 'ASTHYD1', 'BROKENH1', 'CTHLWF1', 'DALNTH01', 'FISHER', 'GUTHEGA', 'HPRG1', 'LK_ECHO', 'CETHANA', 'CHILDSF1', 'HAYMSF1', 'HVGTS', 'SITHE01', 'AGLHAL', 'CHYTWF1', 'DALNTHL1', 'HALLWF2', 'NBHWF1', 'SHGEN', 'HAMISF1', 'HAUGHT11', 'SNOWSTH1', 'BBTHREE3', 'HALLWF1', 'HUMENSW', 'HUMEV', 'SAPHWF1', 'SHPUMP', 'BALDHWF1', 'BBTHREE1', 'MACARTH1']
Processing PUBLIC_DVD_BIDDAYOFFER_D_20210402.feather...
  Found 114 matches with DUID: ['BBTHREE2', 'COHUNSF1', 'HDWF1', 'HDWF2', 'HDWF3', 'HPRL1', 'MACKNTSH', 'SNOWNTH1', 'VSQHT1V1', 'VSSSH1S1', 'W/HOE#1', 'W/HOE#2', 'WHITSF1', 'ASTHYD1', 'BROKENH1', 'CTHLWF1', 'DALNTH01', 'FISHER', 'GUTHEGA', 'HPRG1', 'LK_ECHO', 'CETHANA', 'CHILDSF1', 'HAYMSF1', 'HVGTS', 'SITHE01', 'AGLHAL', 'CHYTWF1', 'DALNTHL1', 'HAL

In [3]:
# Find the first and last settlement dates
first_date = df["SETTLEMENTDATE"].min()
last_date = df["SETTLEMENTDATE"].max()

# Display the results
print(f"First settlement date: {first_date}")
print(f"Last settlement date: {last_date}")

# Optional: Get the full date range
date_range = pd.date_range(start=first_date, end=last_date)
print(f"Total days in range: {len(date_range)}")

First settlement date: 2017-01-01 00:00:00
Last settlement date: 2018-12-31 00:00:00
Total days in range: 730


In [ ]:
# import pandas as pd
# import matplotlib.pyplot as plt
# import numpy as np
# from datetime import datetime

# # List of DUIDs to plot
# duids = ['OSB-AG', 'QPS5', 'PPCCGT', 'TORRB1', 'TORRB4', 'TORRB2', 'TORRB3', 
#          'HDWF2', 'LONSDALE', 'HDWF3', 'HDWF1']

# # Function to create plot for a single DUID
# def plot_duid_bidprices(df, duid):
#     """
#     Create a plot for a specific DUID showing average bid prices by BIDTYPE
    
#     Parameters:
#     df (DataFrame): The dataframe containing the bid data
#     duid (str): The DUID to plot
#     """
#     # Filter for the specific DUID
#     duid_df = df[df["DUID"] == duid].copy()
    
#     if duid_df.empty:
#         print(f"No data found for DUID: {duid}")
#         return None
    
#     # Convert data types for calculations
#     duid_df["BIDPRICE"] = pd.to_numeric(duid_df["BIDPRICE"], errors='coerce')
    
#     # Ensure SETTLEMENTDATE is datetime
#     duid_df["SETTLEMENTDATE"] = pd.to_datetime(duid_df["SETTLEMENTDATE"])
#     duid_df.sort_values(by="SETTLEMENTDATE", inplace=True)
    
#     # Get unique BIDTYPEs for this DUID
#     bid_types = sorted(duid_df["BIDTYPE"].unique())
    
#     # Create figure
#     plt.figure(figsize=(12, 8))
    
#     # Get info about this generator for the title
#     if "Participant" in duid_df.columns and "Station Name" in duid_df.columns:
#         participant = duid_df["Participant"].iloc[0] if not duid_df["Participant"].iloc[0] is None else "Unknown"
#         station = duid_df["Station Name"].iloc[0] if not duid_df["Station Name"].iloc[0] is None else "Unknown"
#         title = f"{duid} - {station} ({participant})"
#     else:
#         title = f"DUID: {duid}"
    
#     # Plot each BIDTYPE
#     for bid_type in bid_types:
#         # Filter for this BIDTYPE
#         bid_df = duid_df[duid_df["BIDTYPE"] == bid_type].copy()
#         bid_df.set_index("SETTLEMENTDATE", inplace=True)
        
#         # Calculate monthly average price
#         monthly_avg = bid_df["BIDPRICE"].resample("M").mean()
        
#         # Skip if all NaN
#         if monthly_avg.isna().all():
#             continue
        
#         # Plot this BIDTYPE
#         plt.plot(
#             monthly_avg.index,
#             monthly_avg,
#             linestyle="-",
#             linewidth=2,
#             marker='o',
#             markersize=4,
#             label=bid_type
#         )
    
#     # Add labels and title
#     plt.title(title, fontsize=14)
#     plt.ylabel("Average Bid Price ($/MWh)", fontsize=12)
#     plt.xlabel("Date", fontsize=12)
    
#     # Format x-axis
#     plt.xticks(rotation=45)
    
#     # Add grid and legend
#     plt.grid(True, linestyle='--', alpha=0.7)
#     plt.legend(loc='best')
    
#     # Adjust layout
#     plt.tight_layout()
    
#     return plt.gcf()

Successfully loaded data with 34633 rows

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34633 entries, 0 to 34632
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   SETTLEMENTDATE  34633 non-null  datetime64[ns]
 1   DUID            34633 non-null  object        
 2   BIDTYPE         34633 non-null  object        
 3   OFFERDATE       34633 non-null  object        
 4   VERSIONNO       34633 non-null  object        
 5   PRICEBAND1      34633 non-null  object        
 6   PRICEBAND2      34633 non-null  object        
 7   PRICEBAND3      34633 non-null  object        
 8   PRICEBAND4      34633 non-null  object        
 9   PRICEBAND5      34633 non-null  object        
 10  PRICEBAND6      34633 non-null  object        
 11  PRICEBAND7      34633 non-null  object        
 12  PRICEBAND8      34633 non-null  object        
 13  PRICEBAND9      34633 non-null  object        
 

In [16]:
df_hpr1.head()

,SETTLEMENTDATE,DUID,BIDTYPE,OFFERDATE,VERSIONNO,PRICEBAND1,PRICEBAND2,PRICEBAND3,PRICEBAND4,PRICEBAND5,...,PRICEBAND7,PRICEBAND8,PRICEBAND9,PRICEBAND10,MINIMUMLOAD,T1,T2,T3,T4,DIRECTION
0,2021-04-01,HPRL1,ENERGY,2021/04/01 03:56:57,1,-979,-450,-175,-90,20,...,135,225,350,500,0,0,0,0,0,None
1,2021-04-01,HPRG1,ENERGY,2021/04/01 03:56:57,1,-983.8,0,54.93,97,169,...,375,998.68,4000,10013.77,0,0,0,0,0,None
2,2021-04-01,HPRL1,RAISEREG,2021/04/01 03:56:57,1,5.51,8.21,12.76,18.41,25.01,...,99.01,269.01,499.01,12000.01,0,None,None,None,None,None
3,2021-04-01,HPRG1,LOWERREG,2021/04/01 03:56:57,1,5.51,7.51,9.26,14.01,25.01,...,99.01,270.01,500.01,12000.01,0,None,None,None,None,None
4,2021-04-01,HPRG1,RAISE5MIN,2021/04/01 03:56:57,1,0,1,2,3,4,...,6,100,750,15000,0,None,None,None,None,None


In [ ]:
import glob
import os
import time
import dask.dataframe as dd
import pandas as pd
from datetime import datetime

def load_valid_parquet_files(folder_pattern):
    """
    Load all valid parquet files matching the given pattern into a Dask DataFrame.
    Excludes hidden files (like macOS ._* files).
    
    Args:
        folder_pattern: Glob pattern to match parquet files
        
    Returns:
        Dask DataFrame containing the data from all valid parquet files
    """
    # Get all files matching the pattern
    all_files = glob.glob(folder_pattern)
    
    # Filter out hidden files (like macOS ._ files)
    valid_files = [f for f in all_files if not os.path.basename(f).startswith('._')]
    
    # Sort files to ensure consistent reading order
    valid_files = sorted(valid_files)
    
    print(f"Found {len(valid_files)} valid parquet files from pattern: {folder_pattern}")
    
    if not valid_files:
        raise ValueError(f"No valid parquet files found for pattern: {folder_pattern}")
    
    # Load files into a Dask DataFrame
    return dd.read_parquet(valid_files, engine='pyarrow')

def filter_and_merge_data(volume_ddf, price_ddf, output_dir):
    """
    Filter data for 2017-2018 and merge volume and price data.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    print("Converting date columns to datetime format...")
    start_time = time.time()
    
    # Convert SETTLEMENTDATE to datetime
    volume_ddf = volume_ddf.assign(
        SETTLEMENTDATE=dd.to_datetime(volume_ddf["SETTLEMENTDATE"], errors="coerce")
    )
    price_ddf = price_ddf.assign(
        SETTLEMENTDATE=dd.to_datetime(price_ddf["SETTLEMENTDATE"], errors="coerce")
    )
    
    print(f"Date conversion defined in {time.time() - start_time:.2f} seconds")
    
    # Filter for 2017-2018 data
    print("Filtering for 2017-2018 data...")
    start_time = time.time()
    
    start_date = pd.Timestamp('2017-01-01')
    end_date = pd.Timestamp('2018-12-31 23:59:59')
    
    volume_filtered = volume_ddf[
        (volume_ddf['SETTLEMENTDATE'] >= start_date) & 
        (volume_ddf['SETTLEMENTDATE'] <= end_date)
    ]
    
    price_filtered = price_ddf[
        (price_ddf['SETTLEMENTDATE'] >= start_date) & 
        (price_ddf['SETTLEMENTDATE'] <= end_date)
    ]
    
    print(f"Filtering defined in {time.time() - start_time:.2f} seconds")
    
    # Persist the filtered DataFrames to avoid recomputation
    print("Persisting filtered data (this may take some time)...")
    start_time = time.time()
    
    volume_filtered = volume_filtered.persist()
    price_filtered = price_filtered.persist()
    
    # Wait for persist to complete
    vol_npartitions = volume_filtered.npartitions
    price_npartitions = price_filtered.npartitions
    
    print(f"Filtered volume data: {vol_npartitions} partitions")
    print(f"Filtered price data: {price_npartitions} partitions")
    print(f"Persistence completed in {time.time() - start_time:.2f} seconds")
    
    # Process one partition at a time
    for i in range(vol_npartitions):
        print(f"Processing volume partition {i+1}/{vol_npartitions}...")
        start_time = time.time()
        
        try:
            # Get one volume partition as pandas DataFrame
            vol_part = volume_filtered.get_partition(i).compute()
            print(f"Loaded volume partition with {len(vol_part)} rows in {time.time() - start_time:.2f} seconds")
            
            # Process each price partition
            for j in range(price_npartitions):
                print(f"  Processing price partition {j+1}/{price_npartitions}...")
                part_start = time.time()
                
                try:
                    # Get one price partition as pandas DataFrame
                    price_part = price_filtered.get_partition(j).compute()
                    print(f"  Loaded price partition with {len(price_part)} rows in {time.time() - part_start:.2f} seconds")
                    
                    # Merge the DataFrames
                    merge_start = time.time()
                    merged_df = vol_part.merge(
                        price_part,
                        on=["SETTLEMENTDATE", "DUID", "BIDTYPE", "BIDBAND"],
                        how="inner",
                        suffixes=('_volume', '_price')
                    )
                    print(f"  Merged to {len(merged_df)} rows in {time.time() - merge_start:.2f} seconds")
                    
                    # If we have merged data, write it to a parquet file
                    if not merged_df.empty:
                        save_start = time.time()
                        output_file = os.path.join(output_dir, f"merged_2017_2018_v{i}_p{j}.parquet")
                        merged_df.to_parquet(output_file, engine='pyarrow', index=False)
                        print(f"  Wrote {len(merged_df)} rows to {output_file} in {time.time() - save_start:.2f} seconds")
                    else:
                        print(f"  No matching data between these partitions")
                    
                except Exception as e:
                    print(f"  Error processing price partition {j}: {str(e)}")
                    continue
                
        except Exception as e:
            print(f"Error processing volume partition {i}: {str(e)}")
            continue

def main():
    # Set up paths for input files
    volume_pattern = "/Volumes/T7/bid-volume-melted-files-A4/*.parquet"
    price_pattern = "/Volumes/T7/bid-price-melted-files-B3/*.parquet"
    
    print("Loading volume data metadata...")
    volume_ddf = load_valid_parquet_files(volume_pattern)
    print(f"Volume data columns: {list(volume_ddf.columns)}")
    print(f"Volume data partitions: {volume_ddf.npartitions}")
    
    print("Loading price data metadata...")
    price_ddf = load_valid_parquet_files(price_pattern)
    print(f"Price data columns: {list(price_ddf.columns)}")
    print(f"Price data partitions: {price_ddf.npartitions}")
    
    # Create output directory if it doesn't exist
    output_dir = "/Volumes/T7/bid-merged-fcas-2017-2018"
    print(f"Will save merged data to {output_dir}...")
    
    # Filter and merge data
    filter_and_merge_data(volume_ddf, price_ddf, output_dir)
    
    # Report completion
    print("All done!")
    print(f"Output saved to: {output_dir}")

if __name__ == "__main__":
    main()

Loading volume data metadata...
Found 1580 valid parquet files from pattern: /Volumes/T7/bid-volume-melted-files-A4/*.parquet
Volume data columns: ['SETTLEMENTDATE', 'DUID', 'BIDTYPE', 'OFFERDATE', 'MAXAVAIL', 'ENABLEMENTMIN', 'ENABLEMENTMAX', 'LOWBREAKPOINT', 'HIGHBREAKPOINT', 'INTERVAL_DATETIME', 'Participant', 'Station Name', 'Region', 'Dispatch Type', 'Category', 'Classification', 'Fuel Source - Primary', 'Fuel Source - Descriptor', 'Technology Type - Primary', 'Technology Type - Descriptor', 'Aggregation', 'BIDBAND', 'BIDVOLUME']
Volume data partitions: 1580
Loading price data metadata...
Found 1452 valid parquet files from pattern: /Volumes/T7/bid-price-melted-files-B3/*.parquet
Price data columns: ['SETTLEMENTDATE', 'DUID', 'BIDTYPE', 'OFFERDATE', 'VERSIONNO', 'MINIMUMLOAD', 'T1', 'T2', 'T3', 'T4', 'BIDBAND', 'BIDPRICE', 'APPLICABLEFROM']
Price data partitions: 1452
Will save merged data to /Volumes/T7/bid-merged-fcas-2017-2018...
Converting date columns to datetime format...
Da